<a href="https://colab.research.google.com/github/ferragina/MyInformationRetrieval/blob/main/7_TagMe_(only).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Entity Linking**

### Initialize TagMe
Please download your own access key to TagMe's service, according to the slides seen in class

In [ ]:
import requests
import json

TAGME_ENDPOINT = "https://tagme.d4science.org/tagme/tag"
LANG = "en" # Also works in italian and german

# this is the key we will be using for REST calls
KEY = "6ab7daea-c174-4254-a9d2-d85f6117bf20-843339462"


## The main function

Now create the function that will "wrap" the REST call. It needs a textual input


In [ ]:
def query_tagme(text):
    payload = {"text": text, "gcube-token": KEY, "lang": LANG}
    # Now we issue a post HTTP request
    r = requests.post(TAGME_ENDPOINT, payload)
    if r.status_code != 200:
        # this means something went wrong with the query
        raise Exception("Error on text: {}\n{}".format(text, r.text))
    return r.json()

## The returned fields

We display the result for a simple textual query. The interesting part, for us, is under the key _annotations_.

This will be a list of annotations containing the following fields:
- **spot (string)**: how the anchor/mention appears in the text.
- **start (int)**: the index of the starting character of the anchor.
- **end (int)**: the index of the ending character of the anchor.
- **link_probability (float ∈[𝟎,𝟏])**: number of times that the spot is an anchor in Wikipedia / number of times the spot occurs in Wikipedia.
- **rho (float ∈[𝟎,𝟏])**: semantic coherency of the linked entity with respect to the query.
- **id (int)**: the Wikipedia identifier of the linked page/entity _(https://en.wikipedia.org/?curid=<>)_.
- **title (string)**: title of the Wikipedia page/entity.

In [ ]:
short_text = "Italy will not be competing in the 2022 world cup"
resp = query_tagme(short_text)
resp

## Handle longer texts

TagME has been designed for handling short texts, but we also have a way to obtain competitive results on longer ones.
This requires modifying the window of spots that are checked by TagME when doing the disambiguation of entities.

Let us therefore consider a slightly longer text stored in a file

In [ ]:
!wget https://raw.githubusercontent.com/LorenzoBellomo/InformationRetrieval/main/data/Leonardo.txt

with open("Leonardo.txt", 'r') as long_file:
    # the file includes a plaintext, so just read it with read()
    text = long_file.read()

text

Now we will change the tagging function we made before, by adding an optional boolean parameter. If true, this means that the text is long, otherwise it is short and we can avoid changing the window.

In [ ]:
def query_tagme(text, long_text=False):
    payload = {"text": text, "gcube-token": KEY, "lang": LANG}
    if long_text:
        # long_text is by defaul false, but if specified by the user, we set the window size at 5
        payload["long_text"] = 5
    r = requests.post(TAGME_ENDPOINT, payload)
    if r.status_code != 200:
        raise Exception("Error on text: {}\n{}".format(text, r.text))
    return r.json()

But how do we deal with noisy annotations?

TagME gives us a "content relevance" score in the form of the **Rho-score**.

We can filter the lowest ranked annotations on relevancy to remove noise. A common threshold for this task is 0.3.

In [ ]:
# Try changing the min_rho parameter and see how it impacts the returned entities
def get_tagme_entities(tagme_response, min_rho=0.3):
    ann = tagme_response["annotations"]
    ann = [a for a in ann if a["rho"] > min_rho] # filter out all the annotations with a rho score lower than the threshold
    return [a["title"] for a in ann if "title" in a] # return just the page titles

Now see which entities _disappear_ when filtering

In [ ]:
print("BEFORE FILTERING")
resp_before = query_tagme(text, long_text=True)
before_filtering = [(a["spot"], a["title"], round(a["rho"]*100)/100) for a in resp_before['annotations'] if "title" in a]

before_filtering

In [ ]:
print("AFTER FILTERING")
resp_after = query_tagme(text, long_text=True)
after_filtering = get_tagme_entities(resp_after)

after_filtering

In [ ]:
print("The annotations that were filtered out are:")
[a for a in before_filtering if a[2] <= 0.3]


Try your own example

In [ ]:
query_tagme("bla bla bla")

# TRY OTHER ANNOTATORS: SWAT

TagME is not the only available entity annotator. There are several more, each one with its own strenghts and weaknesses.

Most of the available annotators are available at [this](https://sobigdata.d4science.org/web/tagme/service-overview) page on the SoBigData Infrastructure

**SWAT** is specifically a salient entity linker, which works best on long, well-constructed texts.
The fields returned are:
- **salience_class (int)**: 1 if the entity is deemed salient, 0 otherwise
- **salience_score (float ∈[𝟎,𝟏])**: the saliency of the enitity in the text (similar to the rho-score in tagme)
- **spans (list)**: list of times where this entity appears, they are described as:
    - *start (int)*: the index of the starting character of the anchor
    - *end (int)*: the index of the ending character of the anchor
- **wiki_id (int)**: the Wikipedia identifier of the page
- **wiki_title (string)**: title of the Wikipedia page

In [ ]:
# this is the new URL of the annotator on the SoBigData Infrastructure
SWAT_ENDPOINT = "https://swat.d4science.org/salience"

# SWAT also requires a title of the content
def query_swat(title, content):
    document = json.dumps({"title": title, "content": content})
    r = requests.post(SWAT_ENDPOINT, data = document, params={'gcube-token': KEY})
    if r.status_code != 200:
        raise Exception("Error on article titled: {}\n{}".format(title, r.text))
    return r.json()["annotations"]

# Print only the first 7 entities
query_swat("Leonardo da Vinci", text)[:7]

## RELATEDNESS
Ok but now that I have entities, how do I deal with them? I need to know which are similar and which are not.
If we don't see any way of "dealing with the entities", how do we unlock its full potential? How is this method more powerful than dealing with generic words as tokens?

There are several ways in which we can obtain the relatedness of couples of entities.
The main one that is shown in this notebook is by querying TagME itself. TagME has an internal relatedness computation framework, so I can ask TagME itself how close two entities are to one another. This metric is computed directly on the Wikipedia Knowledge Graph.

In [ ]:
# The URL where the relatedness is given
ENDPOINT_RELATEDNESS = "https://tagme.d4science.org/tagme/rel"

# In case I need efficiency I can do batch queries of 100 couples per HTTP call
def query_relatedness(e1, e2):
    # Entities require underscores in-place of the spaces. The space is between entity one and entity two
    tt = e1.replace(" ", "_") + " " + e2.replace(" ", "_")
    payload = {"tt": tt, "gcube-token": KEY, "lang": LANG}
    r = requests.post(ENDPOINT_RELATEDNESS, payload)
    if r.status_code != 200:
        raise Exception("Error on relatedness computation: {}\n{}".format(tt, r.text))
    return r.json()

Now let's test the relatedness of three entities.
Two are closely related to one-another (biology and biotechnology).
The last one is completely out of context.

In [ ]:
first = query_relatedness("Biology", "Biotechnology")
second = query_relatedness("Barack Obama", "Biotechnology")
thirds = query_relatedness("Barack Obama", "Biology")
print(first['result'])
print(second['result'])
print(thirds['result'])

#Exercise

Given a phrase, make a query on TagME, and filter out all the annotations with rho score > 0.2

Next exercise, given the next two phrases defined below, make a query on TagME for both of them (in Italian and with rho >= 0.35). Then:
- for each pairs of annotations (one from the first phrase, one from the second) print their relatedness.
- find the pair that maximises this relatedness.

In [ ]:
phrase1 = "Diego Armando Maradona è stato un calciatore, allenatore di calcio e dirigente sportivo \
      argentino di ruolo centrocampista offensivo, campione del mondo nel 1986 e vicecampione del mondo nel 1990 con la \
      nazionale argentina. Soprannominato El Pibe de Oro, è considerato uno dei più grandi calciatori di tutti i tempi. \
      In una carriera da professionista più che ventennale militò nell'Argentinos Juniors, nel Boca Juniors, nel Barcellona, \
       nel Napoli, nel Siviglia e nel Newell's Old Boys."
phrase2 = "Ottavio Bianchi è un ex calciatore e allenatore di calcio italiano, di ruolo centrocampista. \
        Nel 1985 giunse al Milan, dove vinse il primo scudetto della storia partenopea nel campionato 1986-1987, \
        conquistando nella stessa stagione anche la Coppa Italia. Nella stagione successiva Bianchi raggiunse la finale \
        di Coppa Italia (persa contro la Sampdoria) e vinse la Coppa UEFA."
LANG = "it"
